# Live `context_reader` validation

This notebook validates the live Team 04 `context_reader` Grasshopper tool through the Swiftlet MCP bridge.

It checks four things in order:
1. Team 04 path and environment resolution work from the notebook location.
2. The configured LLM settings are loaded and a minimal chat request is attempted.
3. The Rhino or Grasshopper MCP endpoint is reachable and exposes the context-reader tool.
4. A direct MCP call to the live `context_reader` tool returns a parseable result.

Run this notebook while Rhino 8, Grasshopper, and the Swiftlet bridge are already running.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def find_team_root(start: Path) -> Path:
    search_roots = [start, *start.parents]
    for root in search_roots:
        direct_candidate = root
        nested_candidate = root / 'team_04'
        for candidate in (direct_candidate, nested_candidate):
            if (candidate / 'agent').exists() and (candidate / 'PROGRESS.md').exists():
                return candidate.resolve()
    raise RuntimeError('Could not locate the team_04 folder from the current notebook working directory.')


TEAM_ROOT = find_team_root(Path.cwd())
REPO_ROOT = TEAM_ROOT.parent

if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

{
    'team_root': str(TEAM_ROOT),
    'repo_root': str(REPO_ROOT),
    'kernel_cwd': str(Path.cwd()),
}

In [3]:
import sys
from pathlib import Path


def ensure_team_root_on_path() -> Path:
    existing_root = globals().get('TEAM_ROOT')
    if existing_root:
        candidate = Path(existing_root)
        if candidate.exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        direct_candidate = root
        nested_candidate = root / 'team_04'
        for candidate in (direct_candidate, nested_candidate):
            if (candidate / 'agent').exists() and (candidate / 'PROGRESS.md').exists():
                resolved = candidate.resolve()
                if str(resolved) not in sys.path:
                    sys.path.insert(0, str(resolved))
                globals()['TEAM_ROOT'] = resolved
                globals()['REPO_ROOT'] = resolved.parent
                return resolved

    raise RuntimeError('Could not locate the team_04 folder for imports. Run the notebook from this repository.')


TEAM_ROOT = ensure_team_root_on_path()

from agent.config import load_settings

settings = load_settings()

supported_providers = {'openai', 'cloudflare', 'google', 'anthropic', 'local'}
if settings.llm_provider not in supported_providers:
    raise RuntimeError(
        f"Unsupported LLM provider for this notebook: {settings.llm_provider!r}."
    )

connection_summary = {
    'team_root': str(TEAM_ROOT),
    'llm_provider': settings.llm_provider,
    'openai_model': settings.llm_model,
    'mcp_endpoint': settings.mcp_endpoint,
    'request_timeout_seconds': settings.request_timeout_seconds,
    'openai_api_key_configured': bool(settings.api_key),
}

connection_summary

{'team_root': 'C:\\Users\\tuemi\\Downloads\\Glabtools\\IAAC Repo\\bimsc26-datamgmt-session03\\AIA26_Studio\\team_04',
 'llm_provider': 'cloudflare',
 'openai_model': '@cf/qwen/qwen3-30b-a3b-fp8',
 'mcp_endpoint': 'http://localhost:3001/mcp/',
 'request_timeout_seconds': 300.0,
 'openai_api_key_configured': True}

In [6]:
from langchain_openai import ChatOpenAI

llm_status = {
    'provider': settings.llm_provider,
    'ok': False,
    'response': None,
    'error': None,
}

try:
    llm = ChatOpenAI(
        api_key=settings.api_key,
        base_url=settings.base_url,
        model=settings.llm_model,
        timeout=min(settings.request_timeout_seconds, 20.0),
        temperature=0,
        max_retries=0,
    )
    llm_response = llm.invoke('Reply with exactly CONTEXT_READER_LLM_OK')
    llm_text = getattr(llm_response, 'content', llm_response)
    if isinstance(llm_text, list):
        llm_text = ' '.join(str(part) for part in llm_text)
    llm_text = str(llm_text).strip()
    llm_status['response'] = llm_text
    llm_status['ok'] = 'CONTEXT_READER_LLM_OK' in llm_text
except Exception as exc:
    llm_status['error'] = f'{type(exc).__name__}: {exc}'

llm_status

{'provider': 'cloudflare',
 'ok': False,
 'response': None,
 'error': "NotFoundError: Error code: 404 - {'result': None, 'success': False, 'errors': [{'code': 7003, 'message': 'Could not route to /client/v4/accounts/your_cloudflare_account_id/ai/v1/chat/completions, perhaps your object identifier is invalid?'}], 'messages': []}"}

In [5]:
from agent.mcp_client import HttpMcpClient

mcp_client = HttpMcpClient(settings.mcp_endpoint, settings.request_timeout_seconds)
mcp_client.initialize()
discovered_tools = mcp_client.list_tools()
tool_names = sorted(str(tool.get('name', '')).strip() for tool in discovered_tools if tool.get('name'))

preferred_context_tool_name = next(
    (name for name in ('context_reader_04', 'context_reader') if name in tool_names),
    None,
)

mcp_status = {
    'tool_count': len(tool_names),
    'preferred_context_tool_name': preferred_context_tool_name,
    'first_20_tools': tool_names[:20],
}

if preferred_context_tool_name is None:
    raise RuntimeError(
        "Could not find either 'context_reader_04' or 'context_reader' in the live MCP tool list. "
        "Confirm Rhino, Grasshopper, Swiftlet, and the updated Team 04 test_gh definition are all running."
    )

mcp_status

{'tool_count': 4,
 'preferred_context_tool_name': 'context_reader',
 'first_20_tools': ['context_reader',
  'import_building_boundary',
  'site_boundary_reader',
  'test_fit']}

## Direct live tool call

This payload is intentionally simple so the notebook can quickly tell whether the Grasshopper tool is reachable and returning structured data.

If your live `context_reader` expects a different contract, update the payload in the next cell to match the latest Grasshopper input definition.

In [12]:
from agent.mcp_client import HttpMcpClient

mcp_client = HttpMcpClient(settings.mcp_endpoint, settings.request_timeout_seconds)
mcp_client.initialize()

sample_payload = {
    'site_boundary': [[0.0, 0.0], [120.0, 0.0], [120.0, 80.0], [0.0, 80.0], [0.0, 0.0]],
    'roads': [
        [[-20.0, 10.0], [140.0, 10.0]],
        [[-20.0, 65.0], [140.0, 65.0]]
    ],
    'buildings': [
        [[130.0, 20.0], [155.0, 20.0], [155.0, 50.0], [130.0, 50.0], [130.0, 20.0]],
        [[-35.0, 18.0], [-10.0, 18.0], [-10.0, 48.0], [-35.0, 48.0], [-35.0, 18.0]]
    ],
    'entrances': [[60.0, 0.0], [120.0, 40.0]],
    'layer_map': {
        'roads': 'Context::Roads',
        'buildings': 'Context::Buildings',
        'entrances': 'Context::Entrances'
    },
    'analysis_radius_m': 60.0
}

parsed_tool_response = None
raw_tool_response = None
tool_call_error = None

try:
    raw_tool_response = mcp_client.call_tool(preferred_context_tool_name, sample_payload)
    try:
        parsed_tool_response = json.loads(raw_tool_response)
    except json.JSONDecodeError:
        parsed_tool_response = {
            'success': False,
            'raw_response': raw_tool_response,
            'error': 'The tool returned a non-JSON response.',
        }
except Exception as exc:
    tool_call_error = f'{type(exc).__name__}: {exc}'
    parsed_tool_response = {
        'success': False,
        'raw_response': raw_tool_response,
        'error': tool_call_error,
    }

result_summary = {
    'tool_name': preferred_context_tool_name,
    'response_type': type(parsed_tool_response).__name__,
    'top_level_keys': sorted(parsed_tool_response.keys()) if isinstance(parsed_tool_response, dict) else [],
    'success_value': parsed_tool_response.get('success') if isinstance(parsed_tool_response, dict) else None,
    'tool_call_error': tool_call_error,
    'response_preview': parsed_tool_response,
}

result_summary

{'tool_name': 'context_reader',
 'response_type': 'dict',
 'top_level_keys': ['error', 'raw_response', 'success'],
 'success_value': False,
 'tool_call_error': 'ReadTimeout: timed out',
 'response_preview': {'success': False,
  'raw_response': None,
  'error': 'ReadTimeout: timed out'}}

In [13]:
final_check = {
    'llm_connected': llm_status['ok'],
    'llm_provider': llm_status['provider'],
    'mcp_connected': preferred_context_tool_name is not None,
    'context_reader_replied': isinstance(parsed_tool_response, dict),
    'context_reader_success': parsed_tool_response.get('success') if isinstance(parsed_tool_response, dict) else None,
    'context_reader_error': parsed_tool_response.get('error') if isinstance(parsed_tool_response, dict) else None,
    'team_root': str(TEAM_ROOT),
    'tested_tool_name': preferred_context_tool_name,
}

final_check

{'llm_connected': False,
 'llm_provider': 'cloudflare',
 'mcp_connected': True,
 'context_reader_replied': True,
 'context_reader_success': False,
 'context_reader_error': 'ReadTimeout: timed out',
 'team_root': 'C:\\Users\\tuemi\\Downloads\\Glabtools\\IAAC Repo\\bimsc26-datamgmt-session03\\AIA26_Studio\\team_04',
 'tested_tool_name': 'context_reader'}

In [14]:
mcp_client.close()
print('Closed MCP client.')

Closed MCP client.
